# 5th attempt - RNN

In [1]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [2]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
SIZE = 10
AMOUNT_BOARDS = 100000  # 100000 file is corrupted, using 1000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2

In [4]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE, gen)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 3789031
len train:  3069114
len val:  341013
len test:  378904


In [6]:
from functions import build_and_train_rnn

# build and train RNN model
model, history = build_and_train_rnn(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    rnn_units=128,
    dense_units=64,
    epochs=20,
    batch_size=512,
    validation_split=0.2
)

model.summary()

Epoch 1/20
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 63s 12ms/step - accuracy: 0.7584 - loss: 0.4748 - val_accuracy: 0.8008 - val_loss: 0.4271 - learning_rate: 0.0010
Epoch 2/20
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 53s 11ms/step - accuracy: 0.8076 - loss: 0.4067 - val_accuracy: 0.8076 - val_loss: 0.4154 - learning_rate: 0.0010
Epoch 3/20
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 72s 9ms/step - accuracy: 0.8161 - loss: 0.3926 - val_accuracy: 0.8114 - val_loss: 0.4077 - learning_rate: 0.0010
Epoch 4/20
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 51s 11ms/step - accuracy: 0.8210 - loss: 0.3847 - val_accuracy: 0.8256 - val_loss: 0.3856 - learning_rate: 0.0010
Epoch 5/20
4792/4796 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8239 - loss: 0.3803
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 77s 10ms/step - accuracy: 0.8239 - loss: 0.3803 - val_accuracy: 0.8226 - val_loss: 0.3872 - learning_rate: 0.0010
Epoch 6/20
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 54s 11ms/step - accuracy: 

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 376,709 (1.44 MB)

 Trainable params: 125,569 (490.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 251,140 (981.02 KB)

In [7]:
from functions import build_and_train_rnn_v2

# build and train IMPROVED RNN model (Bidirectional stacked LSTM)
model_v2, history_v2 = build_and_train_rnn_v2(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    lstm_units_1=128,
    lstm_units_2=64,
    dense_units_1=128,
    dense_units_2=64,
    dropout_rate=0.3,
    recurrent_dropout=0.2,
    learning_rate=0.001,
    epochs=40,
    batch_size=512,
    validation_split=0.2
)

model_v2.summary()

Epoch 1/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 192s 37ms/step - accuracy: 0.7583 - loss: 0.4816 - val_accuracy: 0.8124 - val_loss: 0.4034 - learning_rate: 0.0010
Epoch 2/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 216s 45ms/step - accuracy: 0.8018 - loss: 0.4151 - val_accuracy: 0.8184 - val_loss: 0.3853 - learning_rate: 0.0010
Epoch 3/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 205s 43ms/step - accuracy: 0.8099 - loss: 0.4005 - val_accuracy: 0.8186 - val_loss: 0.3865 - learning_rate: 0.0010
Epoch 4/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 194s 40ms/step - accuracy: 0.8153 - loss: 0.3939 - val_accuracy: 0.8224 - val_loss: 0.3723 - learning_rate: 0.0010
Epoch 5/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 210s 42ms/step - accuracy: 0.8179 - loss: 0.3896 - val_accuracy: 0.8268 - val_loss: 0.3675 - learning_rate: 0.0010
Epoch 6/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 195s 41ms/step - accuracy: 0.8194 - loss: 0.3862 - val_accuracy: 0.8275 - val_loss: 0.3596 - learning_rate: 0.0010
Epoch 7/40
4796/4796 ━━━━━━━━━━━━━━━━━━━━ 188s 39ms/step -

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 1, 256)         │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 256)         │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,274,117 (4.86 MB)

 Trainable params: 424,449 (1.62 MB)

 Non-trainable params: 768 (3.00 KB)

 Optimizer params: 848,900 (3.24 MB)

In [ ]:
from functions import build_and_train_crnn_v3

# build and train CRNN model (Conv2D + Bidirectional stacked LSTM + AdamW)
model_v3, history_v3 = build_and_train_crnn_v3(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    conv_filters=(32, 64),
    lstm_units_1=128,
    lstm_units_2=64,
    dense_units_1=128,
    dense_units_2=64,
    dropout_rate=0.3,
    recurrent_dropout=0.2,
    learning_rate=0.001,
    weight_decay=1e-4,
    epochs=50,
    batch_size=512,
    validation_split=0.2
)

model_v3.summary()

In [ ]:
# Evaluate CRNN v3
X_test_crnn = X_test_array.reshape((-1, gen-1, SIZE, SIZE, 1)).astype('float32')

test_loss_v3, test_acc_v3 = model_v3.evaluate(X_test_crnn, y_test_array.astype('float32'))
print(f"Test accuracy (v3 CRNN): {test_acc_v3:.3f}")

evaluate_model(model_v3, X_test_crnn, y_test_array, SIZE, gen)

In [8]:
X_test_rnn_v2 = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')

test_loss_v2, test_acc_v2 = model_v2.evaluate(X_test_rnn_v2, y_test_array.astype('float32'))
print(f"Test accuracy (v2): {test_acc_v2:.3f}")

11841/11841 ━━━━━━━━━━━━━━━━━━━━ 37s 3ms/step - accuracy: 0.8367 - loss: 0.3372
Test accuracy (v2): 0.837


In [9]:
evaluate_model(model_v2, X_test_rnn_v2, y_test_array, SIZE, gen)

11841/11841 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │      66994 │      12118 │
│ True=Dead    │      49705 │     250087 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.837
Precision   : 0.574
Recall      : 0.847
F1-score    : 0.684


In [10]:
X_test_rnn = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')

test_loss, test_acc = model.evaluate(X_test_rnn, y_test_array.astype('float32'))
print(f"Test accuracy: {test_acc:.3f}")

11841/11841 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.8293 - loss: 0.3730
Test accuracy: 0.829


In [11]:
evaluate_model(model, X_test_rnn, y_test_array, SIZE, gen)

11841/11841 ━━━━━━━━━━━━━━━━━━━━ 16s 1ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │      66364 │      12748 │
│ True=Dead    │      51874 │     247918 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.829
Precision   : 0.561
Recall      : 0.839
F1-score    : 0.673
